In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import kagglehub


path = kagglehub.dataset_download("saurabhshahane/virus-images")

def find_virus_root(base_path):
    """Recursively finds the directory containing the 'Influenza' folder."""
    for root, dirs, files in os.walk(base_path):
        if "Influenza" in dirs:
            return root
    return None

data_root = find_virus_root(path)

if data_root:
    print(f"✅ Success! Virus data found in: {data_root}")
else:
    print("❌ Error: 'Influenza' folder not found. Check the dataset download.")

    print("Top-level contents:", os.listdir(path))

# Source: https://www.kaggle.com/code/salmankhaliq22/opencv-skimage-denoising-techniques
def preprocess_images(images):
    cleaned_images = []
    for img in images:

        blurred = cv2.GaussianBlur(img, (5, 5), 0)
        cleaned_images.append(blurred)
    return np.array(cleaned_images)


def load_binary_dataset(root_folder, img_size=(64, 64)):
    images = []
    labels = []


    folders = [f for f in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, f))]

    for folder_name in folders:

        label = 0 if folder_name == "Influenza" else 1

        folder_path = os.path.join(root_folder, folder_name)
        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, img_size)
                images.append(img)
                labels.append(label)

    return np.array(images), np.array(labels)


if data_root:
    X, y = load_binary_dataset(data_root)
    print(f"Loaded {len(X)} total images.")
    print(f"Class 0 (Influenza): {np.sum(y == 0)} images")
    print(f"Class 1 (Non-Influenza): {np.sum(y == 1)} images")


    X = preprocess_images(X)

100%|██████████| 2.26G/2.26G [00:33<00:00, 73.2MB/s]

Extracting files...


✅ Success! Virus data found in: /root/.cache/kagglehub/datasets/saurabhshahane/virus-images/versions/1/context_virus_RAW/test
Loaded 252 total images.
Class 0 (Influenza): 19 images
Class 1 (Non-Influenza): 233 images


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train.reshape(len(X_train), -1))
X_test_scaled = scaler.transform(X_test.reshape(len(X_test), -1))


X_train_cnn = X_train_scaled.reshape(-1, 64, 64, 1)
X_test_cnn = X_test_scaled.reshape(-1, 64, 64, 1)
from tensorflow.keras.preprocessing.image import ImageDataGenerator


datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)


datagen.fit(X_train_cnn)

In [ ]:
print(f"Shape of X_train_cnn: {X_train_cnn.shape}")
print(f"Shape of y_train: {y_train.shape}")

Shape of X_train_cnn: (173, 64, 64, 1)
Shape of y_train: (173,)


In [ ]:
# Source: https://www.tensorflow.org/tutorials/images/cnn
# Source: https://keras.io/api/layers/regularization_layers/dropout/
import tensorflow as tf
from tensorflow.keras import models, layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import f1_score, classification_report

def build_model(input_shape=(64, 64, 1)):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

#  Early Stopping
# Source: https://keras.io/api/callbacks/early_stopping/
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model = build_model()
history = model.fit(
    datagen.flow(X_train_cnn, y_train, batch_size=32),
    epochs=50,
    validation_data=(X_test_cnn, y_test),
    callbacks=[early_stopping]
)

# 5. Evaluation (F1-Score)
# Source: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html
y_pred = (model.predict(X_test_cnn) > 0.5).astype(int)
print(f"F1-Score: {f1_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred, target_names=["Control", "Influenza"]))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 190ms/step - accuracy: 0.7803 - loss: 0.6368 - val_accuracy: 0.9737 - val_loss: 0.4537
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step - accuracy: 0.9075 - loss: 0.5620 - val_accuracy: 0.9737 - val_loss: 0.3471
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 159ms/step - accuracy: 0.9075 - loss: 0.3593 - val_accuracy: 0.9737 - val_loss: 0.1155
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 146ms/step - accuracy: 0.9075 - loss: 0.2773 - val_accuracy: 0.9737 - val_loss: 0.1056
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 246ms/step - accuracy: 0.9075 - loss: 0.2159 - val_accuracy: 0.9737 - val_loss: 0.0653
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 281ms/step - accuracy: 0.9075 - loss: 0.1975 - val_accuracy: 0.9737 - val_loss: 0.0643
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 145ms/step - accuracy: 0.9075 - loss: 0.1851 - val_accuracy: 0.9737 - val_loss: 0.0487
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step - accuracy: 0.9075 - loss: 0.1927 - val_accuracy: 0.9737 - val_loss:

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
